# Step 3: Preprocessing dan Data Splitting (Pipeline Terkoreksi)

Notebook ini implementasi pipeline preprocessing yang sudah dikoreksi. Ada dua perbaikan utama dibanding kemungkinan pipeline asli paper:

1. Split dilakukan lebih dulu (70/30, stratified, random_state=42), sebelum langkah apa pun yang mengestimasi parameter dari data.
2. Semua estimator (encoder, seleksi fitur chi-square, scaler) di-fit hanya pada data train, baru diterapkan ke train dan test. Ini penting supaya statistik atau keputusan preprocessing nggak "mengintip" data test.

SMOTE akan diterapkan di notebook selanjutnya (04_modeling_evaluation), juga hanya di train, sesuai yang memang sudah benar dilakukan paper.

In [1]:
import sys, os
sys.path.append('../src')
from data_utils import load_data
from preprocessing import (
    split_raw, encode_features, select_features_chi2, scale_numeric, save_processed,
    NUMERIC_FEATURES, ONEHOT_FEATURES, RAW_CODE_FEATURES, BINARY_FEATURE, CHI2_K, RANDOM_STATE, TEST_SIZE,
)
import pandas as pd

df = load_data()
print('Dataset asli:', df.shape)


Dataset asli: (12330, 18)


## 1. Split Train/Test (70/30, Stratified) — Dilakukan SEBELUM Preprocessing

In [2]:
X_train, X_test, y_train, y_test = split_raw(df)

print(f'Train: {X_train.shape}, Test: {X_test.shape}')
print(f'random_state={RANDOM_STATE}, test_size={TEST_SIZE}')
print()
print('Proporsi kelas positif (Revenue=1):')
print(f'  Train: {y_train.mean()*100:.2f}%')
print(f'  Test:  {y_test.mean()*100:.2f}%')
print(f'  Full:  {df["Revenue"].astype(int).mean()*100:.2f}%')
print()
print('Stratifikasi berhasil menjaga proporsi kelas hampir identik di train dan test.')


Train: (8631, 17), Test: (3699, 17)
random_state=42, test_size=0.3

Proporsi kelas positif (Revenue=1):
  Train: 15.48%
  Test:  15.46%
  Full:  15.47%

Stratifikasi berhasil menjaga proporsi kelas hampir identik di train dan test.


## 2. Encoding: One-Hot untuk `Month` dan `VisitorType` (Fit di Train Saja)

Sesuai temuan di notebook EDA: hanya `Month` dan `VisitorType` yang di-one-hot, sisanya (kode kategori dan Weekend) dipakai apa adanya.

In [3]:
X_train_enc, X_test_enc, encoder = encode_features(X_train, X_test)

print('Jumlah fitur setelah encoding:', X_train_enc.shape[1])
print('(Target: 28 fitur, sesuai klaim paper -- 17 fitur mentah -> 28 setelah one-hot Month & VisitorType)')
print()
print('Kolom train dan test identik?', list(X_train_enc.columns) == list(X_test_enc.columns))
print()
print('Contoh kolom hasil encoding:')
print(list(X_train_enc.columns))


Jumlah fitur setelah encoding: 28
(Target: 28 fitur, sesuai klaim paper -- 17 fitur mentah -> 28 setelah one-hot Month & VisitorType)

Kolom train dan test identik? True

Contoh kolom hasil encoding:
['Administrative', 'Administrative_Duration', 'Informational', 'Informational_Duration', 'ProductRelated', 'ProductRelated_Duration', 'BounceRates', 'ExitRates', 'PageValues', 'SpecialDay', 'OperatingSystems', 'Browser', 'Region', 'TrafficType', 'Weekend', 'Month_Aug', 'Month_Dec', 'Month_Feb', 'Month_Jul', 'Month_June', 'Month_Mar', 'Month_May', 'Month_Nov', 'Month_Oct', 'Month_Sep', 'VisitorType_New_Visitor', 'VisitorType_Other', 'VisitorType_Returning_Visitor']


## 3. Seleksi Fitur Chi-Square (Top-20, Fit HANYA di Train) — Koreksi #2

Paper melakukan seleksi fitur chi-square untuk memilih top-20 dari 28 fitur, tapi tidak menjelaskan apakah ini dilakukan sebelum atau sesudah split. Kalau dihitung dari seluruh dataset (termasuk label test), keputusan fitur mana yang dipakai model sudah "melihat" pola di data test -- ini bentuk data leakage.

Koreksinya: `SelectKBest(chi2, k=20)` di-fit hanya pada data train, lalu daftar fitur yang sama diterapkan ke test.

Satu asumsi tambahan: chi-square dihitung sebelum standard scaling, karena chi2 butuh nilai non-negatif dan semua fitur hasil encoding di sini memang non-negatif secara alami.

In [4]:
X_train_sel, X_test_sel, selected_cols, chi2_scores = select_features_chi2(X_train_enc, y_train, X_test_enc, k=CHI2_K)

print(f'Jumlah fitur terpilih: {len(selected_cols)} (target: {CHI2_K})')
print()
print('20 Fitur terpilih (diurutkan berdasarkan skor chi2, tertinggi ke terendah):')
print(chi2_scores.head(CHI2_K))
print()
print('8 Fitur yang TIDAK terpilih (skor chi2 terendah):')
print(chi2_scores.tail(28 - CHI2_K))


Jumlah fitur terpilih: 20 (target: 20)

20 Fitur terpilih (diurutkan berdasarkan skor chi2, tertinggi ke terendah):
ProductRelated_Duration          592185.350641
PageValues                       127835.808069
Administrative_Duration           33916.508424
Informational_Duration            20987.131790
ProductRelated                    13713.758100
Administrative                      819.170732
Informational                       241.391295
Month_Nov                           152.762369
VisitorType_New_Visitor              76.031270
SpecialDay                           46.039805
Month_May                            43.971559
Month_Mar                            27.819511
BounceRates                          21.216928
ExitRates                            21.207031
Month_Feb                            19.854827
VisitorType_Returning_Visitor        12.186227
Month_Oct                            11.579256
Weekend                               6.597384
Month_Dec                             

## 4. Standard Scaling pada Fitur Numerik (Fit HANYA di Train) — Koreksi #1

Sama halnya dengan seleksi fitur, paper tidak menjelaskan urutan scaling terhadap split. Kalau scaler di-fit dari seluruh dataset, parameter scaling (mean, std) ikut ditentukan data test -- leakage ringan yang bisa membuat evaluasi terlihat lebih bagus dari yang sebenarnya.

Koreksinya: `StandardScaler` di-fit hanya pada train, parameter yang sama dipakai untuk transform train dan test.

In [5]:
X_train_final, X_test_final, scaler, numeric_cols_scaled = scale_numeric(X_train_sel, X_test_sel)

print('Fitur numerik yang di-scale:', numeric_cols_scaled)
print()
print('Statistik setelah scaling (data TRAIN, harus mean~0 std~1):')
print(X_train_final[numeric_cols_scaled].agg(['mean', 'std']).round(3))
print()
print('Statistik setelah scaling (data TEST, TIDAK harus mean~0 karena scaler di-fit dari train saja):')
print(X_test_final[numeric_cols_scaled].agg(['mean', 'std']).round(3))


Fitur numerik yang di-scale: ['Administrative', 'Administrative_Duration', 'Informational', 'Informational_Duration', 'ProductRelated', 'ProductRelated_Duration', 'BounceRates', 'ExitRates', 'PageValues', 'SpecialDay']

Statistik setelah scaling (data TRAIN, harus mean~0 std~1):
      Administrative  Administrative_Duration  Informational  \
mean             0.0                      0.0           -0.0   
std              1.0                      1.0            1.0   

      Informational_Duration  ProductRelated  ProductRelated_Duration  \
mean                    -0.0             0.0                      0.0   
std                      1.0             1.0                      1.0   

      BounceRates  ExitRates  PageValues  SpecialDay  
mean         -0.0        0.0         0.0         0.0  
std           1.0        1.0         1.0         1.0  

Statistik setelah scaling (data TEST, TIDAK harus mean~0 karena scaler di-fit dari train saja):
      Administrative  Administrative_Duration

### Verifikasi Anti-Leakage

Statistik train menunjukkan mean mendekati 0 dan std mendekati 1, sesuai definisi scaling yang di-fit dari data itu sendiri. Statistik test sedikit berbeda dari 0/1 -- justru ini bukti scaler cuma "belajar" dari train. Kalau test juga persis mean 0/std 1, itu tanda scaler ikut di-fit dari test (leakage).

## 5. Simpan Data Terproses untuk Notebook Modeling

In [6]:
OUT_DIR = '../data/processed'
save_processed(X_train_final, X_test_final, y_train, y_test, OUT_DIR)

print('Data terproses disimpan ke:', os.path.abspath(OUT_DIR))
print('Isi folder:', os.listdir(OUT_DIR))
print()
print('Shape final -- X_train:', X_train_final.shape, '| X_test:', X_test_final.shape)


Data terproses disimpan ke: /Users/vickymahfudy/Study/Term 2/Data Science/Replikasi_Paper/data/processed
Isi folder: ['X_train.csv', 'y_train.csv', 'y_test.csv', 'X_test.csv']

Shape final -- X_train: (8631, 20) | X_test: (3699, 20)


## 6. Ringkasan Pipeline

```
Data (12.330 x 18)
  -> Train/Test Split (70/30, stratified, random_state=42)
       [semua langkah berikut fit HANYA di train]
  -> One-hot encoding (Month, VisitorType)          -> 28 fitur
  -> Seleksi fitur chi-square, top-20 dari 28        -> 20 fitur
  -> Standard scaling fitur numerik terpilih
       [lanjut ke notebook 04: SMOTE hanya di train, lalu modeling]
```

Dua koreksi utamanya: scaling dan seleksi fitur chi-square, keduanya sekarang di-fit hanya pada data train, mencegah kemungkinan leakage yang bisa terjadi kalau keduanya dihitung dari seluruh dataset sebelum split.